# NB3 — Flagged-Animal Recovery (side-branch)

**Stage 3 of 3 (optional).** Re-processes the raw recordings of the quality-flagged animals to test whether a clean window can be recovered from an otherwise-noisy recording. Reuses the NB1 pipeline functions (via the shared loader). Produces the recovery tables (`flagged_recovery.csv`, `flagged_rewindowed.csv`, `recovered_windows.csv`, `recovered23_animals_features.csv`) and the recovery/robustness figures used in the recovery appendix.

**Prerequisite:** NB1 (needs `_metadata_merged.csv` and the raw `.txt` files). This notebook is a self-contained analysis branch — the main results in NB2 do **not** depend on it.

## Recover flagged animals (default re-windowing)

*Re-window each flagged recording to find a clean segment → flagged_recovery.csv.*  
<sub>source: `flagged_recovery.py`</sub>

In [ ]:
"""Can any flagged animals be recovered? Scan each flagged recording for a
cleaner sub-window (via the pipeline's quality_window_search + a manual sweep)
and report whether a stretch with RR-CV <= 0.15 exists.
"""
import numpy as np, pandas as pd
from qt_cohort_audit import load_pipeline_ns, animal_id_from_name

ns = load_pipeline_ns(); g = ns.__getitem__
FS = g("FS")
parse = g("parse_header_and_layout"); load = g("load_ecg_file")
findbw = g("find_baseline_window"); bp = g("bandpass_filter"); detect = g("detect_r_peaks")
DATA_DIR = g("DATA_DIR")

m = pd.read_csv("../outputs/_metadata_merged.csv")
m["clean"] = (m.status == "OK") & (m.rr_cv <= 0.15)
flg = m[~m["clean"]].copy().sort_values("rr_cv")
byid = {animal_id_from_name(p.stem): p for p in sorted(DATA_DIR.glob("*.txt"))}

WIN = 10 * FS      # try to find a clean 10-second window
STEP = 2 * FS


def rr_cv_of(sig):
    peaks, inverted, hr, _ = detect(sig)
    if len(peaks) < 5:
        return np.nan, len(peaks)
    rr = np.diff(peaks) / FS * 1000.0
    # physiological RR band for mouse: 80-250 ms; drop impossible gaps
    good = rr[(rr >= 60000.0/700) & (rr <= 60000.0/300)]  # physiological RR (300-700 bpm) matching the pipeline; was loose (60,400)
    if len(good) < 5:
        return np.nan, len(good)
    return good.std() / good.mean(), len(good)


rows = []
for aid in flg.animal_id:
    path = byid.get(int(aid))
    orig_cv = flg.set_index("animal_id").loc[aid, "rr_cv"]
    grp = flg.set_index("animal_id").loc[aid, "group"]
    rec = {"animal_id": int(aid), "group": grp, "orig_rr_cv": round(orig_cv, 2),
           "best_window_cv": np.nan, "recoverable": False}
    if path is not None:
        try:
            n_header, lead, ecg_col, titles = parse(path)
            voltage, markers = load(path, n_header, ecg_col)
            start, end, *_ = findbw(markers, len(voltage))
            if start is not None:
                seg = voltage[start:end]
                filt = bp(seg)
                # sweep sub-windows for the lowest RR-CV
                best = np.inf
                for s0 in range(0, max(1, len(filt) - WIN), STEP):
                    cv, nb = rr_cv_of(filt[s0:s0 + WIN])
                    if not np.isnan(cv):
                        best = min(best, cv)
                if np.isfinite(best):
                    rec["best_window_cv"] = round(best, 2)
                    rec["recoverable"] = best <= 0.15
        except Exception as e:
            rec["note"] = str(e)[:40]
    rows.append(rec)

df = pd.DataFrame(rows)
df.to_csv("../outputs/flagged_recovery.csv", index=False)

n_rec = df.recoverable.sum()
print("RECOVERY SCAN — searching each flagged recording for a clean 10s window\n")
print(df.to_string(index=False))
print("\nSUMMARY:")
print("  recoverable (a clean <=0.15 window exists):  %d / %d" % (n_rec, len(df)))
print("  not recoverable (irregular/noisy throughout): %d / %d" % (len(df) - n_rec, len(df)))
print("\nwrote ../outputs/flagged_recovery.csv")

## Short-window recovery pass

*Retry recovery with shorter windows → flagged_shortwin_recovery.csv (needed by later cells).*  
<sub>source: `flagged_recover_shortwin.py`</sub>

In [ ]:
"""Can the remaining 16 (no clean 10s window) be recovered with a SHORTER window?
Tries 10s / 5s / 3s windows and reports the best achievable RR-CV for each,
so we can see which are borderline-recoverable vs genuinely unusable.
"""
import numpy as np, pandas as pd
from qt_cohort_audit import load_pipeline_ns, animal_id_from_name

ns = load_pipeline_ns(); g = ns.__getitem__
FS = g("FS")
parse = g("parse_header_and_layout"); load = g("load_ecg_file")
findbw = g("find_baseline_window"); bp = g("bandpass_filter"); detect = g("detect_r_peaks")
DATA_DIR = g("DATA_DIR")

rec = pd.read_csv("../outputs/flagged_recovery.csv")
remaining = rec[~rec.recoverable].copy()   # the 16
byid = {animal_id_from_name(p.stem): p for p in sorted(DATA_DIR.glob("*.txt"))}


def best_cv(filt, win_s):
    win = int(win_s * FS); step = int(1 * FS); best = np.inf; nbest = 0
    for s0 in range(0, max(1, len(filt) - win), step):
        w = filt[s0:s0 + win]
        peaks, _, _, _ = detect(w)
        if len(peaks) < 5:
            continue
        rr = np.diff(peaks) / FS * 1000.0
        good = rr[(rr >= 60000.0/700) & (rr <= 60000.0/300)]  # physiological RR (300-700 bpm) matching the pipeline; was loose (60,400)
        if len(good) < 5:
            continue
        cv = good.std() / good.mean()
        if cv < best:
            best = cv; nbest = len(good)
    return (round(best, 3), nbest) if np.isfinite(best) else (np.nan, 0)


rows = []
for _, r in remaining.iterrows():
    aid = int(r.animal_id)
    path = byid.get(aid)
    out = {"animal_id": aid, "group": r.group, "orig_rr_cv": r.orig_rr_cv,
           "cv_10s": np.nan, "cv_5s": np.nan, "cv_3s": np.nan}
    if path is not None:
        try:
            n_header, lead, ecg_col, titles = parse(path)
            voltage, markers = load(path, n_header, ecg_col)
            start, end, *_ = findbw(markers, len(voltage))
            if start is not None:
                filt = bp(voltage[start:end])
                pk, inv, _, _ = detect(filt)
                if inv:
                    filt = -filt
                out["cv_10s"] = best_cv(filt, 10)[0]
                out["cv_5s"] = best_cv(filt, 5)[0]
                out["cv_3s"] = best_cv(filt, 3)[0]
        except Exception:
            pass
    # verdict
    best = np.nanmin([out["cv_5s"], out["cv_3s"]]) if not (np.isnan(out["cv_5s"]) and np.isnan(out["cv_3s"])) else np.nan
    out["verdict"] = ("recoverable @5s" if (out["cv_5s"] <= 0.15) else
                      "recoverable @3s only" if (out["cv_3s"] <= 0.15) else
                      "still not recoverable")
    rows.append(out)

df = pd.DataFrame(rows).sort_values("orig_rr_cv")
df.to_csv("../outputs/flagged_shortwin_recovery.csv", index=False)
print("Trying shorter windows on the 16 not recoverable at 10s:\n")
print(df.to_string(index=False))
n5 = (df.cv_5s <= 0.15).sum()
n3only = ((df.cv_5s > 0.15) & (df.cv_3s <= 0.15)).sum()
none = (df.verdict == "still not recoverable").sum()
print("\nSUMMARY of the 16:")
print("  recoverable with a 5s window:        %d" % n5)
print("  recoverable only with a 3s window:   %d" % n3only)
print("  still not recoverable at any length: %d" % none)

## Re-window sensitivity sweep

*Vary the window length/position; record which animals recover → flagged_rewindowed.csv.*  
<sub>source: `flagged_rewindow_sensitivity.py`</sub>

In [ ]:
"""OPTION 2 — pre-specified re-windowing sensitivity analysis.

Rule (fixed BEFORE looking at any result):
  For each flagged animal, scan the baseline segment left-to-right in 2 s steps
  and take the FIRST 10 s window whose RR-CV (physiological beats only) <= 0.15.
  Extract R-amplitude from that window (template peak-to-peak, matches pipeline).
Then re-run the chronic ethanolamine dose-response on the enlarged sample and
compare with the clean-only result. This is a sensitivity check, NOT the primary.
"""
import numpy as np, pandas as pd
from itertools import combinations
from scipy import stats
from qt_cohort_audit import load_pipeline_ns, animal_id_from_name

ns = load_pipeline_ns(); g = ns.__getitem__
FS = g("FS")
parse = g("parse_header_and_layout"); load = g("load_ecg_file")
findbw = g("find_baseline_window"); bp = g("bandpass_filter")
detect = g("detect_r_peaks"); avg = g("average_beats")
DATA_DIR = g("DATA_DIR")

WIN = 10 * FS
STEP = 2 * FS

m = pd.read_csv("../outputs/_metadata_merged.csv")
m["clean"] = (m.status == "OK") & (m.rr_cv <= 0.15)
flg = m[~m["clean"]].copy()
byid = {animal_id_from_name(p.stem): p for p in sorted(DATA_DIR.glob("*.txt"))}


def first_clean_window_ramp(aid):
    """Pre-specified: first 10s window with RR-CV<=0.15; return (rr_cv, R-amp) or (nan,nan)."""
    path = byid.get(int(aid))
    if path is None:
        return np.nan, np.nan
    try:
        n_header, lead, ecg_col, titles = parse(path)
        voltage, markers = load(path, n_header, ecg_col)
        start, end, *_ = findbw(markers, len(voltage))
        if start is None:
            return np.nan, np.nan
        filt = bp(voltage[start:end])
        pk0, inv, hr0, _ = detect(filt)
        if inv:
            filt = -filt
        for s0 in range(0, max(1, len(filt) - WIN), STEP):
            w = filt[s0:s0 + WIN]
            peaks, _, _, _ = detect(w)
            if len(peaks) < 5:
                continue
            rr = np.diff(peaks) / FS * 1000.0
            good = rr[(rr >= 60000.0/700) & (rr <= 60000.0/300)]  # physiological RR (300-700 bpm) matching the pipeline; was loose (60,400)
            if len(good) < 5:
                continue
            cv = good.std() / good.mean()
            if cv <= 0.15:                      # FIRST qualifying window
                templ, t_ms, beats = avg(w, peaks)
                ramp = float(templ.max() - templ.min())
                return round(cv, 3), round(ramp, 3)
        return np.nan, np.nan
    except Exception:
        return np.nan, np.nan


# recover
rec_rows = []
for aid in flg.animal_id:
    cv, ramp = first_clean_window_ramp(aid)
    if not np.isnan(ramp):
        rec_rows.append({"animal_id": int(aid), "group": flg.set_index("animal_id").loc[aid, "group"],
                         "study": flg.set_index("animal_id").loc[aid, "study"],
                         "recovered_rr_cv": cv, "recovered_r_amp": ramp})
rec = pd.DataFrame(rec_rows)
rec.to_csv("../outputs/flagged_rewindowed.csv", index=False)
print("Recovered %d animals with a pre-specified clean window\n" % len(rec))


def dose_of(gp):
    return 0.0 if gp in ("Control", "Dox") else {"Dox+Etn1.6": 1.6, "Dox+Etn16": 16.0, "Dox+Etn160": 160.0}[gp]


def jt(groups):
    U = 0
    for i, j in combinations(range(len(groups)), 2):
        for a in groups[i]:
            for b in groups[j]:
                U += (b > a) + 0.5 * (b == a)
    nsz = np.array([len(x) for x in groups]); N = nsz.sum()
    mean = (N**2 - (nsz**2).sum())/4; var = (N**2*(2*N+3) - (nsz**2*(2*nsz+3)).sum())/72
    z = (U-mean)/np.sqrt(var); return z, 2*stats.norm.sf(abs(z))


def run_doseresp(df, label):
    df = df[df.study == "chronic"].copy()
    df["dose"] = df["group"].map(dose_of)
    df["ramp"] = pd.to_numeric(df["r_amp"], errors="coerce")
    df = df.dropna(subset=["ramp"])
    groups = [df[df.dose == d]["ramp"].values for d in [0.0, 1.6, 16.0, 160.0]]
    F, p_an = stats.f_oneway(*groups)
    z, p_jt = jt(groups)
    ns_ = [len(x) for x in groups]
    means = [round(x.mean(), 3) for x in groups]
    print("%-28s n=%2d  ANOVA p=%.3f  JT trend p=%.3f  means=%s (grp n=%s)"
          % (label, len(df), p_an, p_jt, means, ns_))


# clean-only R-amp
clean = m[m.clean][["animal_id", "group", "study", "r_amplitude_mv"]].rename(columns={"r_amplitude_mv": "r_amp"})
run_doseresp(clean, "CLEAN ONLY (primary)")

# clean + recovered
rec2 = rec.rename(columns={"recovered_r_amp": "r_amp"})[["animal_id", "group", "study", "r_amp"]]
enlarged = pd.concat([clean, rec2], ignore_index=True)
run_doseresp(enlarged, "CLEAN + RE-WINDOWED (sensitivity)")

print("\nwrote ../outputs/flagged_rewindowed.csv")

## Manual-check recovery (5 s windows)

*The adopted recovery pass with manual verification → recovered_windows.csv (23 recovered).*  
<sub>source: `recovery_manual_check.py`</sub>

In [ ]:
"""How the 23 were recovered + a manual-check aid.
For each recovered animal: find the first clean sub-window (RR-CV <= 0.15) using the
pre-specified rule (10s window; 5s for the 8 that need it), record its exact
start/end time, and plot the FULL recording with that window highlighted so it can
be eyeballed against the raw trace.

Outputs:
  ../outputs/recovered_windows.csv          (animal, window start/end, RR-CV, beats)
  ../outputs/figures/recovered_windows.png  (full trace + highlighted clean window)
"""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from qt_cohort_audit import load_pipeline_ns, animal_id_from_name

ns = load_pipeline_ns(); g = ns.__getitem__
FS = g("FS")
parse = g("parse_header_and_layout"); load = g("load_ecg_file")
findbw = g("find_baseline_window"); bp = g("bandpass_filter"); detect = g("detect_r_peaks")
DATA_DIR = g("DATA_DIR")

m = pd.read_csv("../outputs/_metadata_merged.csv")
rec10 = pd.read_csv("../outputs/flagged_rewindowed.csv")             # 15 @10s
short = pd.read_csv("../outputs/flagged_shortwin_recovery.csv")
rec5 = short[short.cv_5s <= 0.15]                                    # 8 @5s
plan = [(int(a), 10) for a in rec10.animal_id] + [(int(a), 5) for a in rec5.animal_id]
meta = m.set_index("animal_id")
byid = {animal_id_from_name(p.stem): p for p in sorted(DATA_DIR.glob("*.txt"))}


def find_window(aid, win_s):
    path = byid.get(aid)
    if path is None:
        return None
    n_header, lead, ecg_col, titles = parse(path)
    voltage, markers = load(path, n_header, ecg_col)
    start, end, *_ = findbw(markers, len(voltage))
    if start is None:
        return None
    filt = bp(voltage[start:end]); pk, inv, _, _ = detect(filt)
    if inv:
        filt = -filt
    WIN = int(win_s * FS); STEP = int(1 * FS)
    for s0 in range(0, max(1, len(filt) - WIN), STEP):
        w = filt[s0:s0 + WIN]
        peaks, _, _, _ = detect(w)
        if len(peaks) < 5:
            continue
        rr = np.diff(peaks) / FS * 1000.0
        good = rr[(rr >= 60000.0/700) & (rr <= 60000.0/300)]  # physiological RR (300-700 bpm) matching the pipeline; was loose (60,400)
        if len(good) < 5 or good.std()/good.mean() > 0.15:
            continue
        return dict(filt=filt, s0=s0, win=WIN, win_s=win_s,
                    cv=good.std()/good.mean(), beats=len(good),
                    peaks_in=peaks[(peaks >= 0) & (peaks < WIN)])
    return None


rows, panels = [], []
for aid, win_s in plan:
    r = find_window(aid, win_s)
    if r is None:
        continue
    rows.append(dict(Animal=aid, Study=meta.loc[aid, "study"], Group=meta.loc[aid, "group"],
                     window_s=win_s, start_time_s=round(r["s0"]/FS, 1),
                     end_time_s=round((r["s0"]+r["win"])/FS, 1),
                     RR_CV_in_window=round(r["cv"], 3), beats=r["beats"]))
    panels.append((aid, meta.loc[aid, "group"], r))

pd.DataFrame(rows).to_csv("../outputs/recovered_windows.csv", index=False)
print("wrote recovered_windows.csv  (%d animals)" % len(rows))
print(pd.DataFrame(rows).to_string(index=False))

# montage: full trace (grey) with the clean window highlighted (green)
n = len(panels); ncol = 4; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*4.2, nrow*2.0))
axes = np.atleast_1d(axes).ravel()
for ax in axes[n:]:
    ax.axis("off")
for ax, (aid, grp, r) in zip(axes, panels):
    filt = r["filt"]; t = np.arange(len(filt))/FS
    ax.plot(t, filt, lw=0.4, color="#999")
    a, b = r["s0"]/FS, (r["s0"]+r["win"])/FS
    ax.axvspan(a, b, color="#2a9d5c", alpha=0.25)
    seg = filt[r["s0"]:r["s0"]+r["win"]]; ts = np.arange(len(seg))/FS + a
    ax.plot(ts, seg, lw=0.5, color="#1a7a43")
    ax.set_title("animal %d (%s)\nclean window %.0f–%.0fs  RR-CV=%.2f"
                 % (aid, grp.replace("Dox+", ""), a, b, r["cv"]), fontsize=8)
    ax.set_xlabel("time (s)", fontsize=7); ax.tick_params(labelsize=6)
fig.suptitle("How the 23 were recovered: the flagged window is noisy overall (grey),\n"
             "but a regular sub-window (green, RR-CV ≤ 0.15) was found and used for feature extraction",
             fontweight="bold", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("../outputs/figures/recovered_windows.png", dpi=125, bbox_inches="tight")
print("\nsaved recovered_windows.png")

## Feature extraction for recovered animals

*Extract full features for the recovered set → recovered23_animals_features.csv.*  
<sub>source: `recovered23_features.py`</sub>

In [ ]:
"""Full feature extraction for the 23 recovered animals (clean 5s-or-better window).
15 recovered at 10s + 8 more recovered at 5s. Exports Excel + CSV with validity flag.
"""
import numpy as np, pandas as pd
from qt_cohort_audit import load_pipeline_ns, animal_id_from_name

ns = load_pipeline_ns(); g = ns.__getitem__
FS = g("FS")
parse = g("parse_header_and_layout"); load = g("load_ecg_file")
findbw = g("find_baseline_window"); bp = g("bandpass_filter")
detect = g("detect_r_peaks"); avg = g("average_beats")
DATA_DIR = g("DATA_DIR")

m = pd.read_csv("../outputs/_metadata_merged.csv")
rec10 = pd.read_csv("../outputs/flagged_rewindowed.csv")            # 15 @10s
short = pd.read_csv("../outputs/flagged_shortwin_recovery.csv")      # remaining
rec5 = short[short.cv_5s <= 0.15]                                    # 8 more @5s
meta = m.set_index("animal_id")

# build (animal, window_seconds) recovery plan
plan = [(int(a), 10) for a in rec10.animal_id] + [(int(a), 5) for a in rec5.animal_id]
byid = {animal_id_from_name(p.stem): p for p in sorted(DATA_DIR.glob("*.txt"))}


def features(aid, win_s):
    path = byid.get(int(aid))
    if path is None:
        return None
    WIN = int(win_s * FS); STEP = 1 * FS
    n_header, lead, ecg_col, titles = parse(path)
    voltage, markers = load(path, n_header, ecg_col)
    start, end, *_ = findbw(markers, len(voltage))
    if start is None:
        return None
    filt = bp(voltage[start:end])
    _, inv, _, _ = detect(filt)
    if inv:
        filt = -filt
    for s0 in range(0, max(1, len(filt) - WIN), STEP):
        w = filt[s0:s0 + WIN]
        peaks, _, _, _ = detect(w)
        if len(peaks) < 5:
            continue
        rr = np.diff(peaks) / FS * 1000.0
        good = rr[(rr >= 60000.0/700) & (rr <= 60000.0/300)]  # physiological RR (300-700 bpm) matching the pipeline; was loose (60,400)
        if len(good) < 5 or good.std()/good.mean() > 0.15:
            continue
        templ, t_ms, beats = avg(w, peaks)
        base = np.median(templ[(t_ms >= -95) & (t_ms <= -70)]) if ((t_ms >= -95) & (t_ms <= -70)).any() else 0.0

        def amp(lo, hi, mode):
            mk = (t_ms >= lo) & (t_ms <= hi)
            if not mk.any():
                return np.nan
            seg = templ[mk]
            return float(seg.max()-seg.min() if mode == "range" else base-seg.min())
        rr_mean = float(good.mean())
        tmask = (t_ms >= 10) & (t_ms <= min(rr_mean*0.9, 90))
        qt = float(t_ms[tmask][int(np.argmin(templ[tmask]))] + 15) if tmask.any() else np.nan
        qtc = qt/np.sqrt(rr_mean/100.0) if not np.isnan(qt) else np.nan
        return dict(HeartRate_bpm=60000.0/rr_mean, RR_ms=rr_mean, RR_CV=good.std()/rr_mean,
                    Beats=len(good), R_amp_mV=amp(-20, 20, "range"),
                    P_amp_mV=amp(-55, -10, "range"), T_amp_mV=amp(15, 90, "min"),
                    QT_ms=qt, QTc_ms=qtc)
    return None


rows = []
for aid, win_s in plan:
    f = features(aid, win_s)
    rec = {"Animal": aid, "Study": meta.loc[aid, "study"], "Treatment_group": meta.loc[aid, "group"],
           "recovery_window": f"first clean {win_s}s"}
    if f:
        rec.update({k: (round(v, 3) if isinstance(v, float) else v) for k, v in f.items()})
    rows.append(rec)

df = pd.DataFrame(rows)
df["Plausible"] = ((df.R_amp_mV.between(0.1, 1.2)) & (df.HeartRate_bpm.between(250, 800))).map({True: "yes", False: "NO - noisy"})
order = ["Animal", "Study", "Treatment_group", "Plausible", "recovery_window",
         "HeartRate_bpm", "RR_ms", "RR_CV", "Beats", "R_amp_mV", "P_amp_mV", "T_amp_mV", "QT_ms", "QTc_ms"]
df = df[[c for c in order if c in df.columns]].sort_values(["Study", "Treatment_group", "Animal"])
df.to_csv("../outputs/recovered23_animals_features.csv", index=False)
try:
    with pd.ExcelWriter("../outputs/recovered23_animals_features.xlsx", engine="openpyxl") as xl:
        df.to_excel(xl, sheet_name="Recovered_23", index=False)
    print("wrote recovered23_animals_features.xlsx")
except Exception as e:
    print("no xlsx:", e)
print("\n%d recovered animals | plausible: %d | flagged noisy: %d"
      % (len(df), (df.Plausible == "yes").sum(), (df.Plausible != "yes").sum()))
print(df.to_string(index=False))

## Flagged-animal rhythm figures

*Rhythm strips for the flagged recordings (inversion / ectopy inspection).*  
<sub>source: `flagged_rhythm_figures.py`</sub>

In [ ]:
"""Visual proof of WHY the flagged animals were excluded: irregular rhythm.
Reuses NB02's own pipeline functions (via qt_cohort_audit.load_pipeline_ns).

Outputs:
  ../outputs/figures/flagged_rr_montage.png   - RR tachogram per flagged animal
                                                (flat line = regular; spiky = irregular)
  ../outputs/figures/flagged_ecg_examples.png - ECG strips for the 6 worst offenders
"""
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from qt_cohort_audit import load_pipeline_ns, animal_id_from_name

ns = load_pipeline_ns(); g = ns.__getitem__
FS = g("FS")
parse = g("parse_header_and_layout"); load = g("load_ecg_file")
findbw = g("find_baseline_window"); bp = g("bandpass_filter"); detect = g("detect_r_peaks")
DATA_DIR = g("DATA_DIR")

m = pd.read_csv("../outputs/_metadata_merged.csv")
m["clean"] = (m.status == "OK") & (m.rr_cv <= 0.15)
flg = m[~m["clean"]].copy().sort_values("rr_cv", ascending=False)
byid = {animal_id_from_name(p.stem): p for p in sorted(DATA_DIR.glob("*.txt"))}


def process(aid):
    """Return (filtered signal, r-peak indices, rr_ms array) or None."""
    path = byid.get(aid)
    if path is None:
        return None
    try:
        n_header, lead, ecg_col, titles = parse(path)
        voltage, markers = load(path, n_header, ecg_col)
        if voltage.size == 0:
            return None
        start, end, rule, s_t, e_t = findbw(markers, len(voltage))
        if start is None:
            return None
        seg = voltage[start:end]
        filt = bp(seg)
        peaks, inverted, hr, _ = detect(filt)
        if inverted:
            filt = -filt
        if len(peaks) < 3:
            return None
        rr = np.diff(peaks) / FS * 1000.0
        return filt, np.asarray(peaks), rr
    except Exception as e:
        print("  skip animal %s: %s" % (aid, e))
        return None


# gather
data = {}
for aid in flg.animal_id:
    r = process(int(aid))
    if r is not None:
        data[int(aid)] = r
print("processed %d/%d flagged animals" % (len(data), len(flg)))

info = flg.set_index("animal_id")

# ---------- MONTAGE: RR tachogram per animal ----------
ids = [a for a in flg.animal_id if a in data]
n = len(ids); ncol = 5; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*3.1, nrow*2.1))
axes = np.atleast_1d(axes).ravel()
for ax in axes[n:]:
    ax.axis("off")
for ax, aid in zip(axes, ids):
    filt, peaks, rr = data[aid]
    cv = info.loc[aid, "rr_cv"]; grp = info.loc[aid, "group"]
    ax.plot(np.arange(len(rr)), rr, "-o", ms=2.5, lw=0.8, color="#c0392b")
    ax.axhline(np.median(rr), color="#2c3e50", lw=1, ls="--", alpha=0.6)
    ax.set_title("animal %d (%s)\nRR-CV=%.2f" % (aid, grp, cv), fontsize=8.5,
                 color=("#a00" if cv > 0.15 else "#333"))
    ax.set_xlabel("beat #", fontsize=7); ax.set_ylabel("RR (ms)", fontsize=7)
    ax.tick_params(labelsize=6); ax.grid(alpha=.2)
fig.suptitle("Flagged animals — RR-interval tachograms (why they were excluded)\n"
             "A regular rhythm is a flat line; a spiky/variable line is irregular (RR-CV > 0.15 = flagged)",
             fontweight="bold", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("../outputs/figures/flagged_rr_montage.png", dpi=130, bbox_inches="tight")
plt.close(fig)
print("saved flagged_rr_montage.png")

# ---------- EXAMPLES: ECG strips for the 6 most irregular ----------
worst = ids[:6]
fig, axes = plt.subplots(6, 1, figsize=(12, 13))
for ax, aid in zip(axes, worst):
    filt, peaks, rr = data[aid]
    cv = info.loc[aid, "rr_cv"]; grp = info.loc[aid, "group"]
    dur = int(4 * FS)  # 4 seconds
    seg = filt[:dur]; t = np.arange(len(seg)) / FS
    ax.plot(t, seg, lw=0.7, color="#2c3e50")
    pk = peaks[peaks < dur]
    ax.plot(pk/FS, filt[pk], "v", color="#c0392b", ms=7)
    # mark RR gaps
    for i in range(1, len(pk)):
        ax.annotate("", xy=(pk[i]/FS, filt.max()*0.9), xytext=(pk[i-1]/FS, filt.max()*0.9),
                    arrowprops=dict(arrowstyle="<->", color="#888", lw=0.6))
    ax.set_title("animal %d (%s) — RR-CV = %.2f  → irregular rhythm  (uneven beat spacing)"
                 % (aid, grp, cv), fontsize=10, fontweight="bold", color="#a00")
    ax.set_ylabel("mV", fontsize=8); ax.tick_params(labelsize=7); ax.grid(alpha=.2)
axes[-1].set_xlabel("time (s)", fontsize=9)
fig.suptitle("Flagged animals — 4-second ECG strips (6 most irregular)\n"
             "Red markers = detected R-peaks; uneven spacing between them is the irregular rhythm that triggered exclusion",
             fontweight="bold", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("../outputs/figures/flagged_ecg_examples.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print("saved flagged_ecg_examples.png")

## Robustness figure (clean vs full cohort)

*Dose-response trend p-value under clean vs recovered/full inclusion.*  
<sub>source: `robustness_figure.py`</sub>

In [ ]:
"""Three-way robustness figure for the ethanolamine R-amplitude dose-response:
  (1) clean set only        (primary)
  (2) all animals incl. flagged (raw amplitude)
  (3) clean + pre-specified re-windowed
Shows the dose-response is the same however flagged animals are handled.
"""
import numpy as np, pandas as pd
from itertools import combinations
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

m = pd.read_csv("../outputs/_metadata_merged.csv")
m["clean"] = (m.status == "OK") & (m.rr_cv <= 0.15)
rew = pd.read_csv("../outputs/flagged_rewindowed.csv")  # recovered R-amp


def dose_of(gp):
    return 0.0 if gp in ("Control", "Dox") else {"Dox+Etn1.6": 1.6, "Dox+Etn16": 16.0, "Dox+Etn160": 160.0}[gp]


def jt(groups):
    U = 0
    for i, j in combinations(range(len(groups)), 2):
        for a in groups[i]:
            for b in groups[j]:
                U += (b > a) + 0.5 * (b == a)
    nsz = np.array([len(x) for x in groups]); N = nsz.sum()
    mean = (N**2 - (nsz**2).sum())/4; var = (N**2*(2*N+3) - (nsz**2*(2*nsz+3)).sum())/72
    z = (U-mean)/np.sqrt(var); return z, 2*stats.norm.sf(abs(z))


ORDER = [0.0, 1.6, 16.0, 160.0]
LAB = ["No-Etn\n(Ctrl+Dox)", "1.6", "16", "160"]


def prep(df):
    df = df[df.study == "chronic"].copy()
    df["dose"] = df["group"].map(dose_of)
    df["ramp"] = pd.to_numeric(df["ramp"], errors="coerce")
    return df.dropna(subset=["ramp"])


# (1) clean
d1 = m[m.clean][["group", "study", "r_amplitude_mv"]].rename(columns={"r_amplitude_mv": "ramp"})
# (2) all animals (raw amplitude available for all)
d2 = m[["group", "study", "r_amplitude_mv"]].rename(columns={"r_amplitude_mv": "ramp"})
# (3) clean + re-windowed
d3 = pd.concat([d1, rew.rename(columns={"recovered_r_amp": "ramp"})[["group", "study", "ramp"]]], ignore_index=True)

sets = [("Clean set only\n(primary)", prep(d1), "#1f6f8f"),
        ("All animals\n(incl. flagged)", prep(d2), "#c0392b"),
        ("Clean + re-windowed\n(pre-specified)", prep(d3), "#2a9d5c")]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.4), sharey=True)
x = np.arange(4)
summary = []
for ax, (title, df, col) in zip(axes, sets):
    groups = [df[df.dose == d]["ramp"].values for d in ORDER]
    means = [g.mean() for g in groups]; sems = [g.std()/np.sqrt(len(g)) for g in groups]
    nsz = [len(g) for g in groups]
    z, pjt = jt(groups); F, pan = stats.f_oneway(*groups)
    summary.append((title.replace("\n", " "), sum(nsz), pan, pjt))
    for i, g in enumerate(groups):
        jx = i + np.random.RandomState(i).uniform(-0.1, 0.1, len(g))
        ax.scatter(jx, g, s=28, color=col, edgecolor="k", linewidth=0.3, alpha=0.55, zorder=2)
    ax.errorbar(x, means, yerr=sems, fmt="o-", color=col, lw=2.4, ms=10, capsize=5,
                zorder=3, markeredgecolor="k", markeredgewidth=0.5)
    ax.set_title("%s\nn=%d   ANOVA p=%.3f   trend p=%.3f" % (title, sum(nsz), pan, pjt),
                 fontweight="bold", fontsize=11,
                 color=("#1a5e1a" if pjt < 0.05 else "#333"))
    ax.set_xticks(x); ax.set_xticklabels(LAB, fontsize=9)
    ax.set_xlabel("ethanolamine dose (mg/kg)")
    ax.grid(alpha=.2)
    for i, nn in enumerate(nsz):
        ax.text(i, ax.get_ylim()[0], "n=%d" % nn, ha="center", va="bottom", fontsize=7, color="#666")
axes[0].set_ylabel("R-wave amplitude (mV)", fontsize=11)

fig.suptitle("Robustness of the ethanolamine dose-response to how flagged animals are handled (chronic study)\n"
             "The R-amplitude decline and its trend significance hold across all three approaches",
             fontweight="bold", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.9])
fig.savefig("../outputs/figures/robustness_flagged_handling.png", dpi=150, bbox_inches="tight")
print("saved robustness_flagged_handling.png\n")
print("%-38s | n  | ANOVA p | JT trend p" % "approach")
for t, n, pa, pj in summary:
    print("%-38s | %2d | %.3f   | %.3f" % (t, n, pa, pj))

## Dose-group figure with recovered animals

*Dose-group amplitudes including the recovered recordings.*  
<sub>source: `dose_group_recovered_figure.py`</sub>

In [ ]:
"""Dose grouping (ethanolamine dose) WITH recovered animals included.
Chronic R-wave amplitude: clean animals + pre-specified re-windowed recoveries,
recovered points drawn distinctly. Shows the dose-response holds with recovery.
"""
import numpy as np, pandas as pd
from itertools import combinations
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

m = pd.read_csv("../outputs/_metadata_merged.csv")
m["clean"] = (m.status == "OK") & (m.rr_cv <= 0.15)
rew = pd.read_csv("../outputs/flagged_rewindowed.csv")


def dose_of(gp):
    return 0.0 if gp in ("Control", "Dox") else {"Dox+Etn1.6": 1.6, "Dox+Etn16": 16.0, "Dox+Etn160": 160.0}[gp]


def jt(groups):
    U = 0
    for i, j in combinations(range(len(groups)), 2):
        for a in groups[i]:
            for b in groups[j]:
                U += (b > a) + 0.5 * (b == a)
    nsz = np.array([len(x) for x in groups]); N = nsz.sum()
    mean = (N**2 - (nsz**2).sum())/4; var = (N**2*(2*N+3) - (nsz**2*(2*nsz+3)).sum())/72
    z = (U-mean)/np.sqrt(var); return z, 2*stats.norm.sf(abs(z))


ORDER = [0.0, 1.6, 16.0, 160.0]
LAB = ["No-Etn\n(Ctrl+Dox)", "1.6", "16", "160"]

# clean chronic
cl = m[m.clean & (m.study == "chronic")][["group", "r_amplitude_mv"]].copy()
cl["dose"] = cl["group"].map(dose_of); cl["ramp"] = pd.to_numeric(cl["r_amplitude_mv"], errors="coerce")
cl = cl.dropna(subset=["ramp"]); cl["src"] = "clean"

# recovered chronic
rc = rew[rew.study == "chronic"].copy()
rc["dose"] = rc["group"].map(dose_of); rc["ramp"] = rc["recovered_r_amp"]; rc["src"] = "recovered"

both = pd.concat([cl[["dose", "ramp", "src"]], rc[["dose", "ramp", "src"]]], ignore_index=True)
groups = [both[both.dose == d]["ramp"].values for d in ORDER]
means = [g.mean() for g in groups]; sems = [g.std()/np.sqrt(len(g)) for g in groups]
z, pjt = jt(groups); F, pan = stats.f_oneway(*groups)
nrec = (rc.dose.isin(ORDER)).sum()

fig, ax = plt.subplots(figsize=(9, 6.3))
x = np.arange(4)
for i, d in enumerate(ORDER):
    cvals = cl[cl.dose == d]["ramp"].values
    rvals = rc[rc.dose == d]["ramp"].values
    jx = i + np.random.RandomState(i).uniform(-0.11, 0.11, len(cvals))
    ax.scatter(jx, cvals, s=45, color="#1f6f8f", edgecolor="k", linewidth=0.3,
               alpha=0.7, zorder=2, label="clean animals" if i == 0 else None)
    jr = i + np.random.RandomState(i+7).uniform(-0.11, 0.11, len(rvals))
    ax.scatter(jr, rvals, s=95, color="#e08e0b", edgecolor="k", linewidth=0.6,
               marker="D", alpha=0.95, zorder=4, label="recovered animals" if i == 0 else None)
ax.errorbar(x, means, yerr=sems, fmt="o-", color="#2c3e50", lw=2.5, ms=12, capsize=6,
            zorder=3, label="mean ± SEM (all)")
ax.annotate("", xy=(3, means[3]), xytext=(0, means[0]),
            arrowprops=dict(arrowstyle="-|>", color="#c0392b", lw=2.2, ls="--", alpha=0.7))

ax.set_title("Dose grouping WITH recovered animals — R-wave amplitude (chronic)\n"
             "n = %d (%d clean + %d recovered)   ·   ANOVA p = %.3f   ·   trend p = %.3f"
             % (len(both), len(cl), nrec, pan, pjt), fontweight="bold", fontsize=12,
             color="#1a5e1a" if pjt < 0.05 else "#333")
ax.set_xticks(x); ax.set_xticklabels(LAB)
ax.set_xlabel("ethanolamine dose (mg/kg)", fontsize=11)
ax.set_ylabel("R-wave amplitude (mV)", fontsize=11)
ax.legend(fontsize=10, loc="upper right"); ax.grid(alpha=.2)
ax.text(0.5, -0.15, "Recovered animals (orange diamonds) fall in line with the dose-response — the trend holds",
        transform=ax.transAxes, ha="center", fontsize=9.5, style="italic", color="#555")
fig.tight_layout()
fig.savefig("../outputs/figures/dose_group_recovered.png", dpi=150, bbox_inches="tight")
print("saved dose_group_recovered.png | n=%d ANOVA p=%.3f trend p=%.3f" % (len(both), pan, pjt))